# Multi-Base Model Experiments

**Goal**: Test if CAGP's decomposition insight generalizes to other KGE architectures.

Base Models:
1. **DistMult** (bilinear, symmetric)
2. **TransE** (translation-based)
3. **ComplEx** (complex embeddings)

**Memory**: ~4GB GPU for FB15k-237

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset
import json
import os
import random

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Using device: cuda
GPU: Tesla T4
Memory: 15.8 GB


In [2]:
CONFIG = {
    'epochs': 50,
    'embedding_dim': 100,
    'batch_size': 2048,
    'lr': 0.001,
    'margin': 1.0,  # For TransE
    'kl_weight': 0.01,
    'seeds': [42, 123, 456],
}

In [3]:
import os

DATA_DIR = 'data/raw'
DATASET = 'fb15k-237'  # 'wn18rr' for smaller

# Download FB15K-237 if not present (for Colab)
if not os.path.exists(f'{DATA_DIR}/{DATASET}/train.txt'):
    print("Downloading FB15K-237 from Hugging Face...")
    os.makedirs(f'{DATA_DIR}/{DATASET}', exist_ok=True)
    !pip install -q datasets
    from datasets import load_dataset
    ds = load_dataset("KGraph/FB15k-237")

    for split, filename in [('train', 'train.txt'), ('validation', 'valid.txt'), ('test', 'test.txt')]:
        with open(f'{DATA_DIR}/{DATASET}/{filename}', 'w') as f:
            for row in ds[split]:
                # Dataset has 'text' column with space-separated h/r/t
                parts = row['text'].split()
                if len(parts) >= 3:
                    f.write(f"{parts[0]}\t{parts[1]}\t{parts[2]}\n")
    print("Downloaded FB15K-237")

def load_triples(path):
    triples = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                triples.append((parts[0], parts[1], parts[2]))
    return triples

train = load_triples(f'{DATA_DIR}/{DATASET}/train.txt')
test = load_triples(f'{DATA_DIR}/{DATASET}/test.txt')

entities = set()
relations = set()
for h, r, t in train + test:
    entities.add(h)
    entities.add(t)
    relations.add(r)

ent2idx = {e: i for i, e in enumerate(entities)}
rel2idx = {r: i for i, r in enumerate(relations)}

print(f"Dataset: {DATASET}")
print(f"Entities: {len(entities)}, Relations: {len(relations)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train.txt:   0%|          | 0.00/21.3M [00:00<?, ?B/s]

valid.txt: 0.00B [00:00, ?B/s]

test.txt: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/272115 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/17535 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20466 [00:00<?, ? examples/s]

Downloaded FB15K-237
Dataset: fb15k-237
Entities: 14534, Relations: 237


## 1. Base Model Definitions

In [4]:
class DistMultGP(nn.Module):
    """DistMult with GP-style variance."""

    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.num_entities = num_entities
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))

    def forward(self, h, r, t):
        if self.training:
            h_emb = self._sample(h)
            t_emb = self._sample(t)
        else:
            h_emb = self.entity_mean[h]
            t_emb = self.entity_mean[t]
        r_emb = self.relation_emb(r)
        return (h_emb * r_emb * t_emb).sum(dim=-1)

    def _sample(self, idx):
        mean = self.entity_mean[idx]
        std = torch.exp(0.5 * self.entity_logvar[idx])
        return mean + std * torch.randn_like(std)

    def get_gp_uncertainty(self, h, t):
        h_var = torch.exp(self.entity_logvar[h]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[t]).mean(dim=-1)
        return (h_var + t_var) / 2

    def get_coverage_uncertainty(self, h, r, t):
        return 2.0 - self.coverage[h, r] - self.coverage[t, r]

    def get_uncertainty(self, h, r, t):
        gp = self.get_gp_uncertainty(h, t)
        cov = self.get_coverage_uncertainty(h, r, t)
        gp_norm = gp / (gp.mean() + 1e-8) * cov.mean()
        alpha = torch.sigmoid(self.alpha_logit)
        return alpha * gp_norm + (1 - alpha) * cov

    def precompute_coverage(self, triples, ent2idx, rel2idx):
        for h, r, t in triples:
            self.coverage[ent2idx[h], rel2idx[r]] = 1.0
            self.coverage[ent2idx[t], rel2idx[r]] = 1.0

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities


class TransEGP(nn.Module):
    """TransE with GP-style variance: h + r ≈ t."""

    def __init__(self, num_entities, num_relations, dim, margin=1.0):
        super().__init__()
        self.num_entities = num_entities
        self.margin = margin
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))

    def forward(self, h, r, t):
        if self.training:
            h_emb = self._sample(h)
            t_emb = self._sample(t)
        else:
            h_emb = self.entity_mean[h]
            t_emb = self.entity_mean[t]
        r_emb = self.relation_emb(r)
        # TransE score: negative distance
        return -torch.norm(h_emb + r_emb - t_emb, p=2, dim=-1)

    def _sample(self, idx):
        mean = self.entity_mean[idx]
        std = torch.exp(0.5 * self.entity_logvar[idx])
        return mean + std * torch.randn_like(std)

    def get_gp_uncertainty(self, h, t):
        h_var = torch.exp(self.entity_logvar[h]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[t]).mean(dim=-1)
        return (h_var + t_var) / 2

    def get_coverage_uncertainty(self, h, r, t):
        return 2.0 - self.coverage[h, r] - self.coverage[t, r]

    def get_uncertainty(self, h, r, t):
        gp = self.get_gp_uncertainty(h, t)
        cov = self.get_coverage_uncertainty(h, r, t)
        gp_norm = gp / (gp.mean() + 1e-8) * cov.mean()
        alpha = torch.sigmoid(self.alpha_logit)
        return alpha * gp_norm + (1 - alpha) * cov

    def precompute_coverage(self, triples, ent2idx, rel2idx):
        for h, r, t in triples:
            self.coverage[ent2idx[h], rel2idx[r]] = 1.0
            self.coverage[ent2idx[t], rel2idx[r]] = 1.0

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities


class ComplExGP(nn.Module):
    """ComplEx with GP-style variance: complex embeddings."""

    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.num_entities = num_entities
        # Real and imaginary parts
        self.entity_re_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_im_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_re_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.entity_im_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_re = nn.Embedding(num_relations, dim)
        self.relation_im = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.relation_re.weight)
        nn.init.xavier_uniform_(self.relation_im.weight)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))

    def forward(self, h, r, t):
        if self.training:
            h_re, h_im = self._sample(h)
            t_re, t_im = self._sample(t)
        else:
            h_re, h_im = self.entity_re_mean[h], self.entity_im_mean[h]
            t_re, t_im = self.entity_re_mean[t], self.entity_im_mean[t]

        r_re, r_im = self.relation_re(r), self.relation_im(r)

        # ComplEx scoring
        score = (h_re * r_re * t_re).sum(dim=-1) + \
                (h_im * r_re * t_im).sum(dim=-1) + \
                (h_re * r_im * t_im).sum(dim=-1) - \
                (h_im * r_im * t_re).sum(dim=-1)
        return score

    def _sample(self, idx):
        re_mean = self.entity_re_mean[idx]
        im_mean = self.entity_im_mean[idx]
        re_std = torch.exp(0.5 * self.entity_re_logvar[idx])
        im_std = torch.exp(0.5 * self.entity_im_logvar[idx])
        return (re_mean + re_std * torch.randn_like(re_std),
                im_mean + im_std * torch.randn_like(im_std))

    def get_gp_uncertainty(self, h, t):
        h_var = (torch.exp(self.entity_re_logvar[h]).mean(dim=-1) +
                 torch.exp(self.entity_im_logvar[h]).mean(dim=-1)) / 2
        t_var = (torch.exp(self.entity_re_logvar[t]).mean(dim=-1) +
                 torch.exp(self.entity_im_logvar[t]).mean(dim=-1)) / 2
        return (h_var + t_var) / 2

    def get_coverage_uncertainty(self, h, r, t):
        return 2.0 - self.coverage[h, r] - self.coverage[t, r]

    def get_uncertainty(self, h, r, t):
        gp = self.get_gp_uncertainty(h, t)
        cov = self.get_coverage_uncertainty(h, r, t)
        gp_norm = gp / (gp.mean() + 1e-8) * cov.mean()
        alpha = torch.sigmoid(self.alpha_logit)
        return alpha * gp_norm + (1 - alpha) * cov

    def precompute_coverage(self, triples, ent2idx, rel2idx):
        for h, r, t in triples:
            self.coverage[ent2idx[h], rel2idx[r]] = 1.0
            self.coverage[ent2idx[t], rel2idx[r]] = 1.0

    def kl_loss(self):
        kl_re = -0.5 * torch.sum(1 + self.entity_re_logvar - self.entity_re_mean.pow(2) - self.entity_re_logvar.exp())
        kl_im = -0.5 * torch.sum(1 + self.entity_im_logvar - self.entity_im_mean.pow(2) - self.entity_im_logvar.exp())
        return (kl_re + kl_im) / self.num_entities

## 2. Training & Evaluation

In [5]:
def train_model(model, triples, ent2idx, rel2idx, epochs, use_margin=False):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'])

    heads = torch.tensor([ent2idx[h] for h, r, t in triples])
    relations = torch.tensor([rel2idx[r] for h, r, t in triples])
    tails = torch.tensor([ent2idx[t] for h, r, t in triples])

    loader = DataLoader(
        TensorDataset(heads, relations, tails),
        batch_size=CONFIG['batch_size'], shuffle=True
    )

    criterion = nn.BCEWithLogitsLoss()

    model.train()
    for epoch in range(epochs):
        for batch_h, batch_r, batch_t in loader:
            batch_h = batch_h.to(device)
            batch_r = batch_r.to(device)
            batch_t = batch_t.to(device)

            pos = model(batch_h, batch_r, batch_t)
            neg_t = torch.randint(0, len(ent2idx), batch_t.shape, device=device)
            neg = model(batch_h, batch_r, neg_t)

            if use_margin:
                # Margin-based ranking loss for TransE
                loss = F.relu(CONFIG['margin'] - pos + neg).mean()
            else:
                loss = criterion(pos, torch.ones_like(pos)) + criterion(neg, torch.zeros_like(neg))

            loss += CONFIG['kl_weight'] * model.kl_loss()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}/{epochs}")

    return model


def evaluate_auroc(model, test_triples, ent2idx, rel2idx):
    model.eval()

    heads = torch.tensor([ent2idx.get(h, 0) for h, r, t in test_triples]).to(device)
    relations = torch.tensor([rel2idx.get(r, 0) for h, r, t in test_triples]).to(device)
    tails = torch.tensor([ent2idx.get(t, 0) for h, r, t in test_triples]).to(device)

    results = {}

    with torch.no_grad():
        ood_tails = torch.randint(0, len(ent2idx), tails.shape, device=device)

        # CAGP (combined)
        id_unc = model.get_uncertainty(heads, relations, tails).cpu().numpy()
        ood_unc = model.get_uncertainty(heads, relations, ood_tails).cpu().numpy()
        labels = np.concatenate([np.zeros(len(id_unc)), np.ones(len(ood_unc))])
        scores = np.concatenate([id_unc, ood_unc])
        results['cagp'] = roc_auc_score(labels, scores)

        # GP-only
        id_gp = model.get_gp_uncertainty(heads, tails).cpu().numpy()
        ood_gp = model.get_gp_uncertainty(heads, ood_tails).cpu().numpy()
        scores_gp = np.concatenate([id_gp, ood_gp])
        results['gp_only'] = roc_auc_score(labels, scores_gp)

        # Coverage-only
        id_cov = model.get_coverage_uncertainty(heads, relations, tails).cpu().numpy()
        ood_cov = model.get_coverage_uncertainty(heads, relations, ood_tails).cpu().numpy()
        scores_cov = np.concatenate([id_cov, ood_cov])
        results['coverage_only'] = roc_auc_score(labels, scores_cov)

    return results

## 3. Main Experiment

In [6]:
MODELS = {
    'DistMult': (DistMultGP, False),
    'TransE': (TransEGP, True),  # use_margin=True
    'ComplEx': (ComplExGP, False),
}

all_results = {model_name: {'cagp': [], 'gp_only': [], 'coverage_only': []} for model_name in MODELS}

for seed in CONFIG['seeds']:
    print(f"\n{'='*60}")
    print(f"Seed {seed}")
    print('='*60)

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    for model_name, (ModelClass, use_margin) in MODELS.items():
        print(f"\n  {model_name}...")

        model = ModelClass(len(ent2idx), len(rel2idx), CONFIG['embedding_dim'])
        model.precompute_coverage(train, ent2idx, rel2idx)
        model = train_model(model, train, ent2idx, rel2idx, CONFIG['epochs'], use_margin)

        results = evaluate_auroc(model, test, ent2idx, rel2idx)

        for method in results:
            all_results[model_name][method].append(results[method])

        print(f"    GP-only: {results['gp_only']:.4f}")
        print(f"    Coverage-only: {results['coverage_only']:.4f}")
        print(f"    CAGP: {results['cagp']:.4f}")
        print(f"    Alpha: {torch.sigmoid(model.alpha_logit).item():.4f}")

        del model
        torch.cuda.empty_cache()


Seed 42

  DistMult...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    GP-only: 0.7527
    Coverage-only: 0.8219
    CAGP: 0.9601
    Alpha: 0.5000

  TransE...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    GP-only: 0.7758
    Coverage-only: 0.8221
    CAGP: 0.9631
    Alpha: 0.5000

  ComplEx...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    GP-only: 0.7559
    Coverage-only: 0.8202
    CAGP: 0.9594
    Alpha: 0.5000

Seed 123

  DistMult...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    GP-only: 0.7536
    Coverage-only: 0.8201
    CAGP: 0.9591
    Alpha: 0.5000

  TransE...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    GP-only: 0.7759
    Coverage-only: 0.8218
    CAGP: 0.9629
    Alpha: 0.5000

  ComplEx...
    Epoch 10/50
    Epoch 20/50
    Epoch 30/50
    Epoch 40/50
    Epoch 50/50
    GP-only:

In [7]:
print("\n" + "="*70)
print(f"FINAL RESULTS: {DATASET.upper()}")
print("="*70)

print(f"\n{'Model':<12} {'GP-only':<15} {'Coverage-only':<15} {'CAGP':<15} {'Synergy':<10}")
print("-"*67)

for model_name in MODELS:
    gp = np.mean(all_results[model_name]['gp_only'])
    cov = np.mean(all_results[model_name]['coverage_only'])
    cagp = np.mean(all_results[model_name]['cagp'])
    synergy = cagp - max(gp, cov)

    print(f"{model_name:<12} {gp:.4f}          {cov:.4f}          {cagp:.4f}          +{synergy:.4f}")

print("\n--- Key Insight ---")
print("If CAGP > max(GP-only, Coverage-only) for all models, the decomposition generalizes.")


FINAL RESULTS: FB15K-237

Model        GP-only         Coverage-only   CAGP            Synergy   
-------------------------------------------------------------------
DistMult     0.7526          0.8205          0.9594          +0.1389
TransE       0.7752          0.8219          0.9630          +0.1411
ComplEx      0.7553          0.8211          0.9599          +0.1388

--- Key Insight ---
If CAGP > max(GP-only, Coverage-only) for all models, the decomposition generalizes.


In [8]:
# Save results
output = {
    'dataset': DATASET,
    'config': CONFIG,
    'results': {
        model_name: {
            method: {'mean': float(np.mean(v)), 'std': float(np.std(v))}
            for method, v in methods.items()
        }
        for model_name, methods in all_results.items()
    }
}

with open(f'multi_model_results_{DATASET}.json', 'w') as f:
    json.dump(output, f, indent=2)

print(f"\nResults saved to multi_model_results_{DATASET}.json")


Results saved to multi_model_results_fb15k-237.json
